In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import time
import threading

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def simple_resample(audio, orig_sr, target_sr):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1)
    return resampled

def play_audio_to_separate_outputs(file_path, with_test_tones=False):
    """Play audio file routing left channel to outputs 1/2 and right channel to outputs 3/4"""
    debug_print("KOMPLETE AUDIO DIRECT PLAYBACK:")
    
    # Get device list
    devices = sd.query_devices()
    
    # Look for Komplete Audio ASIO devices
    output_12_id = None
    output_34_id = None
    
    for i, device in enumerate(devices):
        if device['max_output_channels'] > 0:
            # Check if it's an ASIO device
            if 'hostapi' in device and device['hostapi'] == 3:  # ASIO
                name = device['name'].lower()
                debug_print(f"Found ASIO device {i}: {device['name']}", 1)
                if "output 1/2" in name and ("komplete" in name or "1/2" in name):
                    output_12_id = i
                    debug_print(f"Found Output 1/2 ASIO device: ID {i} - {device['name']}", 1)
                elif "output 3/4" in name and ("komplete" in name or "3/4" in name):
                    output_34_id = i
                    debug_print(f"Found Output 3/4 ASIO device: ID {i} - {device['name']}", 1)
    
    # Check if we found the devices
    if output_12_id is None or output_34_id is None:
        debug_print("Could not find both ASIO devices for Komplete Audio", 1)
        debug_print("Looking for any suitable multi-channel device instead", 1)
        
        # Try to find any multi-channel device as a fallback
        for i, device in enumerate(devices):
            if device['max_output_channels'] >= 4 and 'hostapi' in device and device['hostapi'] == 3:
                debug_print(f"Found multi-channel ASIO device: {device['name']}", 1)
                output_12_id = i  # Use the same device for both outputs
                output_34_id = i
                break
        
        if output_12_id is None:
            debug_print("No suitable audio device found.", 1)
            return False
    
    debug_print(f"Using Output 1/2: Device {output_12_id}", 1)
    debug_print(f"Using Output 3/4: Device {output_34_id}", 1)
    
    try:
        # Load file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Get device sample rate
        try:
            device_info = sd.query_devices(output_12_id)
            device_sr = int(device_info['default_samplerate'])
        except:
            device_sr = 44100  # Default to 44.1kHz if query fails
        
        debug_print(f"Using sample rate: {device_sr}Hz", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Split channels
        left_channel = audio[:, 0].copy()
        right_channel = audio[:, 1].copy()
        
        # Add test signals to beginning if requested
        if with_test_tones:
            test_duration = 5  # seconds
            test_samples = int(test_duration * device_sr)
            
            t = np.linspace(0, test_duration, test_samples, endpoint=False)
            test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
            test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
            
            # Add fade in/out
            fade_samples = int(0.1 * device_sr)
            fade_in = np.linspace(0, 1, fade_samples)
            fade_out = np.linspace(1, 0, fade_samples)
            
            test_left[:fade_samples] *= fade_in
            test_left[-fade_samples:] *= fade_out
            test_right[:fade_samples] *= fade_in
            test_right[-fade_samples:] *= fade_out
            
            # Replace beginning of channels with test tones
            if len(left_channel) > test_samples:
                left_channel[:test_samples] = test_left
                right_channel[:test_samples] = test_right
                debug_print("Added test tones: 440Hz on left, 880Hz on right", 1)
        
        # Create stereo output for each device (we need to send stereo)
        left_output = np.column_stack((left_channel, left_channel)).astype(np.float32)
        right_output = np.column_stack((right_channel, right_channel)).astype(np.float32)
        
        # Start playback
        debug_print("\nSTARTING PLAYBACK:")
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ PLAYING NOW!", 1)
        
        # Play through separate threads
        def play_left():
            try:
                sd.play(left_output, device_sr, device=output_12_id, blocking=True)
                debug_print("Output 1/2 playback complete", 1)
            except Exception as e:
                debug_print(f"Error on Output 1/2: {e}", 1)
        
        def play_right():
            try:
                sd.play(right_output, device_sr, device=output_34_id, blocking=True)
                debug_print("Output 3/4 playback complete", 1)
            except Exception as e:
                debug_print(f"Error on Output 3/4: {e}", 1)
        
        thread_left = threading.Thread(target=play_left)
        thread_right = threading.Thread(target=play_right)
        
        thread_left.start()
        thread_right.start()
        
        # Show progress
        total_duration = len(audio) / device_sr
        start_time = time.time()
        
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1)
        
        try:
            while thread_left.is_alive() or thread_right.is_alive():
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                time.sleep(5)  # Update every 5 seconds for long files
        except KeyboardInterrupt:
            sd.stop()
            debug_print("Playback interrupted by user", 1)
        
        debug_print("Playback complete!", 1)
        return True
        
    except Exception as e:
        debug_print(f"Error in audio playback: {e}", 1)
        return False

if __name__ == "__main__":
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch\participant_00_combined.wav"
    
    debug_print("KOMPLETE AUDIO 6 CHANNEL ROUTING PLAYBACK")
    debug_print("========================================")
    debug_print(f"AUDIO FILE: {file_path}")
    
    # Play the file (set second parameter to True to include test tones)
    success = play_audio_to_separate_outputs(file_path, False)
    
    if success:
        debug_print("✓ PLAYBACK SUCCESSFUL!")
    else:
        debug_print("⚠️ PLAYBACK FAILED")
        debug_print("Make sure your Komplete Audio 6 is properly connected and ASIO drivers are installed.")

[21:26:39] KOMPLETE AUDIO 6 CHANNEL ROUTING PLAYBACK
[21:26:39] ========================================
[21:26:39] AUDIO FILE: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch\participant_00_combined.wav
[21:26:39] KOMPLETE AUDIO DIRECT PLAYBACK:
[21:26:39]   Found ASIO device 28: Speakers (Nahimic mirroring Wave Speaker)
[21:26:39]   Found ASIO device 29: Speakers (Realtek HD Audio output with SST)
[21:26:39]   Found ASIO device 33: Headphones (Realtek HD Audio 2nd output with SST)
[21:26:39]   Found ASIO device 34: Speakers ()
[21:26:39]   Found ASIO device 35: Speakers (Nahimic Easy Surround)
[21:26:39]   Found ASIO device 39: Speakers ()
[21:26:39]   Found ASIO device 40: SPDIF Output (SPDIF Output)
[21:26:39]   Found ASIO device 41: Output 3/4 (Output 3/4)
[21:26:39]   Found Output 3/4 ASIO device: ID 41 - Output 3/4 (Output 3/4)
[21:26:39]   Found ASIO device 42: Output 1/2 (Output 1/2)
[21:26:39]   Found Output 1/2 ASIO device: ID

In [1]:
 completion flags
        left_complete = threading.Event()
        right_complete = threading.Event()
        
        # Function to check for errors in a thread
        def play_with_error_check(output, device_id, event, channel_name):
            try:
                # Use blocksize for lower latency
                sd.play(output, device_sr, device=device_id, blocksize=buffer_size, blocking=False)
                
                # Wait for completion
                sd_event = sd.get_stream().active
                while sd_event:
                    time.sleep(0.1)  # Check every 100ms
                    try:
                        sd_event = sd.get_stream().active
                    except:
                        break
                        
                debug_print(f"{channel_name} playback complete", 1, text_widget)
            except Exception as e:
                debug_print(f"Error on {channel_name}: {e}", 1, text_widget)
            finally:
                event.set()  # Signal completion regardless of success or failure
        
        # Play through separate threads
        thread_left = threading.Thread(
            target=play_with_error_check, 
            args=(left_output, output_12_id, left_complete, "Output 1/2")
        )
        
        thread_right = threading.Thread(
            target=play_with_error_check, 
            args=(right_output, output_34_id, right_complete, "Output 3/4")
        )
        
        # Start threads nearly simultaneously for better sync
        thread_left.start()
        thread_right.start()
        
        # Show progress
        total_duration = len(audio) / device_sr
        start_time = time.time()
        
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1, text_widget)
        
        # Show progress without blocking the GUI
        playback_active = True
        
        def update_progress():
            if not playback_active:
                return
                
            elapsed = time.time() - start_time
            percent = min(100, int(elapsed / total_duration * 100))
            debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1, text_widget)
            
            # Schedule next update if playback is still active
            if (not left_complete.is_set() or not right_complete.is_set()) and elapsed < total_duration + 2:
                # Update more frequently than before (every 2 seconds instead of 5)
                if text_widget and text_widget.winfo_exists():
                    text_widget.after(2000, update_progress)
        
        # Start progress updates
        if text_widget:
            text_widget.after(1000, update_progress)
        
        # Wait for both threads to complete (for non-GUI usage)
        if not text_widget:
            try:
                while not left_complete.is_set() or not right_complete.is_set():
                    elapsed = time.time() - start_time
                    percent = min(100, int(elapsed / total_duration * 100))
                    debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1, text_widget)
                    time.sleep(2)  # Update every 2 seconds
            except KeyboardInterrupt:
                playback_active = False
                sd.stop()
                debug_print("Playback interrupted by user", 1, text_widget)
        
        debug_print("Playback started successfully", 1, text_widget)
        return True
        
    except Exception as e:
        debug_print(f"Error in audio playback: {e}", 1, text_widget)
        return False

# GUI Application
class AudioPlayerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Participant Audio Player")
        self.root.geometry("800x600")
        
        # Set up main frame
        main_frame = ttk.Frame(root, padding="10")
        main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Title label
        title_label = ttk.Label(main_frame, text="Breathing Space Audio Player", font=("Arial", 16, "bold"))
        title_label.pack(pady=10)
        
        # Frame for participant selection
        select_frame = ttk.LabelFrame(main_frame, text="Select Participant", padding="10")
        select_frame.pack(fill=tk.X, pady=10)
        
        # Participant listbox with scrollbar
        list_frame = ttk.Frame(select_frame)
        list_frame.pack(fill=tk.BOTH, expand=True)
        
        self.participant_listbox = tk.Listbox(list_frame, height=10, exportselection=False)
        self.participant_listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        scrollbar = ttk.Scrollbar(list_frame, orient=tk.VERTICAL, command=self.participant_listbox.yview)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.participant_listbox.config(yscrollcommand=scrollbar.set)
        
        # Playback options
        options_frame = ttk.LabelFrame(main_frame, text="Playback Options", padding="10")
        options_frame.pack(fill=tk.X, pady=10)
        
        # Buffer size selection
        buffer_frame = ttk.Frame(options_frame)
        buffer_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(buffer_frame, text="Buffer Size:").pack(side=tk.LEFT, padx=5)
        
        self.buffer_size = tk.StringVar(value="512")
        buffer_sizes = ["128", "256", "512", "1024", "2048"]
        buffer_combo = ttk.Combobox(buffer_frame, textvariable=self.buffer_size, values=buffer_sizes, width=10)
        buffer_combo.pack(side=tk.LEFT, padx=5)
        
        # Test tone checkbox
        self.test_tones = tk.BooleanVar(value=False)
        test_check = ttk.Checkbutton(options_frame, text="Include test tones at start", variable=self.test_tones)
        test_check.pack(anchor=tk.W, pady=5)
        
        # Play button
        self.play_button = ttk.Button(main_frame, text="Play Selected Audio", command=self.play_audio)
        self.play_button.pack(pady=10)
        
        # Log area
        log_frame = ttk.LabelFrame(main_frame, text="Log", padding="10")
        log_frame.pack(fill=tk.BOTH, expand=True, pady=10)
        
        self.log_text = tk.Text(log_frame, height=10, wrap=tk.WORD)
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        log_scrollbar = ttk.Scrollbar(log_frame, orient=tk.VERTICAL, command=self.log_text.yview)
        log_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.log_text.config(yscrollcommand=log_scrollbar.set)
        
        # Status bar
        self.status_var = tk.StringVar(value="Ready")
        status_bar = ttk.Label(root, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W)
        status_bar.pack(side=tk.BOTTOM, fill=tk.X)
        
        # Load participant files
        self.load_participant_files()
    
    def load_participant_files(self):
        """Load participant files from the directory"""
        self.folder_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
        self.participant_files = []
        
        debug_print(f"Loading participant files from {self.folder_path}", 0, self.log_text)
        
        try:
            # Get all WAV files
            all_files = [f for f in os.listdir(self.folder_path) if f.endswith('.wav')]
            
            # Sort by participant number
            def extract_participant_num(filename):
                match = re.search(r'participant_(\d+)_combined\.wav', filename)
                if match:
                    try:
                        return int(match.group(1))
                    except:
                        return 999  # High number for sorting if can't convert
                return 999
            
            all_files.sort(key=extract_participant_num)
            self.participant_files = all_files
            
            # Add to listbox
            self.participant_listbox.delete(0, tk.END)
            for file in self.participant_files:
                self.participant_listbox.insert(tk.END, file)
            
            debug_print(f"Loaded {len(all_files)} participant files", 0, self.log_text)
            
        except Exception as e:
            debug_print(f"Error loading participant files: {e}", 0, self.log_text)
            messagebox.showerror("Error", f"Could not load participant files: {e}")
    
    def play_audio(self):
        """Play the selected participant audio file"""
        selection = self.participant_listbox.curselection()
        
        if not selection:
            messagebox.showinfo("Selection Required", "Please select a participant file to play.")
            return
        
        selected_file = self.participant_files[selection[0]]
        file_path = os.path.join(self.folder_path, selected_file)
        
        # Get buffer size
        try:
            buffer_size = int(self.buffer_size.get())
        except:
            buffer_size = 512  # Default
        
        # Clear log
        self.log_text.delete(1.0, tk.END)
        debug_print(f"Playing file: {selected_file}", 0, self.log_text)
        debug_print(f"Buffer size: {buffer_size}", 0, self.log_text)
        
        # Update status
        self.status_var.set(f"Playing: {selected_file}")
        self.play_button.config(state=tk.DISABLED)
        
        # Play in a separate thread to not freeze GUI
        def play_thread():
            success = play_audio_to_separate_outputs(
                file_path, 
                self.log_text, 
                buffer_size=buffer_size, 
                test_tones=self.test_tones.get()
            )
            
            # Re-enable button when done (in main thread)
            self.root.after(0, lambda: self.play_button.config(state=tk.NORMAL))
            self.root.after(0, lambda: self.status_var.set("Ready" if success else "Playback Error"))
        
        threading.Thread(target=play_thread, daemon=True).start()

# Run the application
if __name__ == "__main__":
    root = tk.Tk()
    app = AudioPlayerApp(root)
    root.mainloop()

[21:00:03] Loading participant files from C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch
[21:00:03] Loaded 100 participant files


In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import time
import threading
import tkinter as tk
from tkinter import ttk, messagebox
import os
import re

# Configure sounddevice for low latency
sd.default.latency = 'low'

class AudioPlayer:
    def __init__(self):
        self.audio_data = None
        self.sample_rate = 0
        self.playback_thread = None
        self.is_playing = False
        self.is_paused = False
        self.current_position = 0
        self.output_12_id = None
        self.output_34_id = None
        self.left_stream = None
        self.right_stream = None
        self.stop_event = threading.Event()
        self.pause_event = threading.Event()

    def load_file(self, file_path, text_widget=None):
        """Load audio file and prepare for playback"""
        try:
            debug_print(f"Loading file: {file_path}", 1, text_widget)
            audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
            debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1, text_widget)
            
            self.audio_data = audio
            self.sample_rate = file_sr
            self.current_position = 0
            return True
        except Exception as e:
            debug_print(f"Error loading audio file: {e}", 1, text_widget)
            return False

    def prepare_playback(self, text_widget=None, buffer_size=512, test_tones=False):
        """Prepare audio devices and data for playback"""
        # Get device list
        devices = sd.query_devices()
        
        # Look for Komplete Audio ASIO devices
        self.output_12_id = None
        self.output_34_id = None
        
        for i, device in enumerate(devices):
            if device['max_output_channels'] > 0:
                # Check if it's an ASIO device
                if 'hostapi' in device and device['hostapi'] == 3:  # ASIO
                    name = device['name'].lower()
                    debug_print(f"Found ASIO device {i}: {device['name']}", 1, text_widget)
                    if "output 1/2" in name and ("komplete" in name or "1/2" in name):
                        self.output_12_id = i
                        debug_print(f"Found Output 1/2 ASIO device: ID {i} - {device['name']}", 1, text_widget)
                    elif "output 3/4" in name and ("komplete" in name or "3/4" in name):
                        self.output_34_id = i
                        debug_print(f"Found Output 3/4 ASIO device: ID {i} - {device['name']}", 1, text_widget)
        
        # Check if we found the devices
        if self.output_12_id is None or self.output_34_id is None:
            debug_print("Could not find both ASIO devices for Komplete Audio", 1, text_widget)
            debug_print("Looking for any suitable multi-channel device instead", 1, text_widget)
            
            # Try to find any multi-channel device as a fallback
            for i, device in enumerate(devices):
                if device['max_output_channels'] >= 4 and 'hostapi' in device and device['hostapi'] == 3:
                    debug_print(f"Found multi-channel ASIO device: {device['name']}", 1, text_widget)
                    self.output_12_id = i  # Use the same device for both outputs
                    self.output_34_id = i
                    break
            
            if self.output_12_id is None:
                debug_print("No suitable audio device found.", 1, text_widget)
                return False
        
        # Get device sample rate
        try:
            device_info = sd.query_devices(self.output_12_id)
            device_sr = int(device_info['default_samplerate'])
        except:
            device_sr = 44100  # Default to 44.1kHz if query fails
        
        debug_print(f"Using sample rate: {device_sr}Hz", 1, text_widget)
        debug_print(f"Using buffer size: {buffer_size} samples", 1, text_widget)
        
        # Resample if needed
        if self.sample_rate != device_sr:
            debug_print(f"Sample rate mismatch: file={self.sample_rate}Hz, device={device_sr}Hz", 1, text_widget)
            self.audio_data = simple_resample(self.audio_data, self.sample_rate, device_sr, text_widget)
            self.sample_rate = device_sr
        
        # Add test signals to beginning if requested
        if test_tones:
            test_duration = 3  # seconds
            test_samples = int(test_duration * device_sr)
            
            t = np.linspace(0, test_duration, test_samples, endpoint=False)
            test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
            test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
            
            # Add fade in/out
            fade_samples = int(0.1 * device_sr)
            fade_in = np.linspace(0, 1, fade_samples)
            fade_out = np.linspace(1, 0, fade_samples)
            
            test_left[:fade_samples] *= fade_in
            test_left[-fade_samples:] *= fade_out
            test_right[:fade_samples] *= fade_in
            test_right[-fade_samples:] *= fade_out
            
            # Replace beginning of channels with test tones
            if len(self.audio_data) > test_samples:
                audio_copy = self.audio_data.copy()
                audio_copy[:test_samples, 0] = test_left
                audio_copy[:test_samples, 1] = test_right
                self.audio_data = audio_copy
                debug_print("Added test tones: 440Hz on left, 880Hz on right", 1, text_widget)
        
        return True

    def start_playback(self, text_widget=None, buffer_size=512, start_time=0.0):
        """Start audio playback from specified time position"""
        if not self.audio_data is not None or self.is_playing:
            return False
            
        # Calculate start position in samples
        start_sample = int(start_time * self.sample_rate)
        if start_sample >= len(self.audio_data):
            start_sample = 0
            
        self.current_position = start_sample
        
        # Reset events
        self.stop_event.clear()
        self.pause_event.clear()
        
        # Split channels
        left_channel = self.audio_data[:, 0].copy()
        right_channel = self.audio_data[:, 1].copy()
        
        # Create stereo output for each device (we need to send stereo)
        left_output = np.column_stack((left_channel, left_channel)).astype(np.float32)
        right_output = np.column_stack((right_channel, right_channel)).astype(np.float32)
        
        # Start the playback thread
        self.playback_thread = threading.Thread(
            target=self._playback_thread_func,
            args=(left_output, right_output, buffer_size, start_sample, text_widget),
            daemon=True
        )
        
        self.is_playing = True
        self.is_paused = False
        self.playback_thread.start()
        
        debug_print(f"Playback started from position {start_time:.2f}s", 1, text_widget)
        return True
        
    def _playback_thread_func(self, left_output, right_output, buffer_size, start_sample, text_widget):
        """Thread function for audio playback"""
        try:
            # Truncate outputs to start at the desired position
            left_output = left_output[start_sample:]
            right_output = right_output[start_sample:]
            
            # Set up callback functions for streaming
            def left_callback(outdata, frames, time, status):
                if self.stop_event.is_set():
                    raise sd.CallbackStop
                
                if self.pause_event.is_set():
                    outdata.fill(0)
                    return
                    
                position = self.current_position
                end = position + frames
                
                if end > len(left_output):
                    outdata[:len(left_output) - position] = left_output[position:]
                    outdata[len(left_output) - position:] = 0
                    raise sd.CallbackStop
                else:
                    outdata[:] = left_output[position:end]
                    self.current_position = end
            
            def right_callback(outdata, frames, time, status):
                if self.stop_event.is_set():
                    raise sd.CallbackStop
                
                if self.pause_event.is_set():
                    outdata.fill(0)
                    return
                    
                position = self.current_position
                end = position + frames
                
                if end > len(right_output):
                    outdata[:len(right_output) - position] = right_output[position:]
                    outdata[len(right_output) - position:] = 0
                    raise sd.CallbackStop
                else:
                    outdata[:] = right_output[position:end]
            
            # Start the streams
            self.left_stream = sd.OutputStream(
                device=self.output_12_id,
                channels=2,
                callback=left_callback,
                samplerate=self.sample_rate,
                blocksize=buffer_size,
                dtype='float32'
            )
            
            self.right_stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=right_callback,
                samplerate=self.sample_rate,
                blocksize=buffer_size,
                dtype='float32'
            )
            
            self.left_stream.start()
            self.right_stream.start()
            
            # Wait for completion
            while self.left_stream.active or self.right_stream.active:
                time.sleep(0.1)
                if self.stop_event.is_set():
                    break
            
        except Exception as e:
            debug_print(f"Error in playback thread: {e}", 1, text_widget)
        
        finally:
            # Clean up
            if hasattr(self, 'left_stream') and self.left_stream:
                self.left_stream.stop()
                self.left_stream.close()
                self.left_stream = None
                
            if hasattr(self, 'right_stream') and self.right_stream:
                self.right_stream.stop()
                self.right_stream.close()
                self.right_stream = None
                
            self.is_playing = False
            self.is_paused = False
            
            # Signal in main thread that playback is complete
            if text_widget and text_widget.winfo_exists():
                text_widget.after(0, lambda: debug_print("Playback complete", 1, text_widget))
    
    def pause_playback(self, text_widget=None):
        """Pause audio playback"""
        if self.is_playing and not self.is_paused:
            self.pause_event.set()
            self.is_paused = True
            debug_print("Playback paused", 1, text_widget)
            return True
        return False
    
    def resume_playback(self, text_widget=None):
        """Resume audio playback"""
        if self.is_playing and self.is_paused:
            self.pause_event.clear()
            self.is_paused = False
            debug_print("Playback resumed", 1, text_widget)
            return True
        return False
    
    def stop_playback(self, text_widget=None):
        """Stop audio playback"""
        if self.is_playing:
            self.stop_event.set()
            
            # Close streams
            if hasattr(self, 'left_stream') and self.left_stream:
                self.left_stream.stop()
                
            if hasattr(self, 'right_stream') and self.right_stream:
                self.right_stream.stop()
                
            # Wait for thread to finish
            if self.playback_thread and self.playback_thread.is_alive():
                self.playback_thread.join(timeout=1.0)
                
            self.is_playing = False
            self.is_paused = False
            debug_print("Playback stopped", 1, text_widget)
            return True
        return False

    def get_position(self):
        """Get current playback position in seconds"""
        if self.sample_rate > 0:
            return self.current_position / self.sample_rate
        return 0.0
    
    def get_duration(self):
        """Get audio duration in seconds"""
        if self.audio_data is not None and self.sample_rate > 0:
            return len(self.audio_data) / self.sample_rate
        return 0.0

def debug_print(message, indent=0, text_widget=None):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    log_message = f"[{timestamp}] {indent_str}{message}"
    print(log_message)
    
    # If we have a text widget, also log there
    if text_widget:
        text_widget.insert(tk.END, log_message + "\n")
        text_widget.see(tk.END)  # Auto-scroll to end

def simple_resample(audio, orig_sr, target_sr, text_widget=None):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1, text_widget)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1, text_widget)
    return resampled

class AudioPlayerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Breathing Space Audio Player")
        self.root.geometry("600x500")  # More compact size
        
        # Create audio player instance
        self.audio_player = AudioPlayer()
        
        # Create main frame with padding
        self.main_frame = ttk.Frame(root, padding="5")
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Create a style
        self.style = ttk.Style()
        self.style.configure("TButton", padding=2)
        self.style.configure("TFrame", padding=2)
        
        # Create UI
        self._create_participant_frame()
        self._create_playback_controls()
        self._create_options_frame()
        self._create_log_area()
        
        # Status bar
        self.status_var = tk.StringVar(value="Ready")
        self.status_bar = ttk.Label(root, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)
        
        # Load participant files
        self.load_participant_files()
        
        # Setup progress updating
        self.update_progress_id = None
    
    def _create_participant_frame(self):
        """Create the participant selection area"""
        select_frame = ttk.LabelFrame(self.main_frame, text="Participant Selection", padding="5")
        select_frame.pack(fill=tk.X, pady=5)
        
        # Participant dropdown
        ttk.Label(select_frame, text="Select Participant:").grid(row=0, column=0, padx=5, pady=2, sticky=tk.W)
        
        self.participant_var = tk.StringVar()
        self.participant_dropdown = ttk.Combobox(select_frame, textvariable=self.participant_var, state="readonly", width=40)
        self.participant_dropdown.grid(row=0, column=1, padx=5, pady=2, sticky=tk.W+tk.E)
    
    def _create_playback_controls(self):
        """Create playback controls section"""
        controls_frame = ttk.LabelFrame(self.main_frame, text="Playback Controls", padding="5")
        controls_frame.pack(fill=tk.X, pady=5)
        
        # Time input
        time_frame = ttk.Frame(controls_frame)
        time_frame.grid(row=0, column=0, columnspan=2, padx=5, pady=2, sticky=tk.W)
        
        ttk.Label(time_frame, text="Start Time (mm:ss):").pack(side=tk.LEFT, padx=5)
        
        self.minutes_var = tk.StringVar(value="00")
        self.seconds_var = tk.StringVar(value="00")
        
        minutes_entry = ttk.Entry(time_frame, textvariable=self.minutes_var, width=3)
        minutes_entry.pack(side=tk.LEFT)
        
        ttk.Label(time_frame, text=":").pack(side=tk.LEFT)
        
        seconds_entry = ttk.Entry(time_frame, textvariable=self.seconds_var, width=3)
        seconds_entry.pack(side=tk.LEFT, padx=(0, 10))
        
        # Progress display
        self.progress_var = tk.StringVar(value="00:00 / 00:00")
        progress_label = ttk.Label(time_frame, textvariable=self.progress_var)
        progress_label.pack(side=tk.LEFT, padx=10)
        
        # Control buttons
        button_frame = ttk.Frame(controls_frame)
        button_frame.grid(row=1, column=0, columnspan=2, padx=5, pady=5, sticky=tk.W+tk.E)
        
        self.play_button = ttk.Button(button_frame, text="Play", command=self.start_playback, width=8)
        self.play_button.pack(side=tk.LEFT, padx=2)
        
        self.pause_button = ttk.Button(button_frame, text="Pause", command=self.pause_resume, width=8)
        self.pause_button.pack(side=tk.LEFT, padx=2)
        self.pause_button.config(state=tk.DISABLED)
        
        self.stop_button = ttk.Button(button_frame, text="Stop", command=self.stop_playback, width=8)
        self.stop_button.pack(side=tk.LEFT, padx=2)
        self.stop_button.config(state=tk.DISABLED)
        
        self.restart_button = ttk.Button(button_frame, text="Restart", command=self.restart_playback, width=8)
        self.restart_button.pack(side=tk.LEFT, padx=2)
        self.restart_button.config(state=tk.DISABLED)
    
    def _create_options_frame(self):
        """Create playback options section"""
        options_frame = ttk.LabelFrame(self.main_frame, text="Playback Options", padding="5")
        options_frame.pack(fill=tk.X, pady=5)
        
        # Buffer size
        buffer_frame = ttk.Frame(options_frame)
        buffer_frame.pack(fill=tk.X)
        
        ttk.Label(buffer_frame, text="Buffer Size:").pack(side=tk.LEFT, padx=5)
        
        self.buffer_size = tk.StringVar(value="512")
        buffer_sizes = ["128", "256", "512", "1024", "2048"]
        buffer_combo = ttk.Combobox(buffer_frame, textvariable=self.buffer_size, values=buffer_sizes, width=8, state="readonly")
        buffer_combo.pack(side=tk.LEFT, padx=5)
        
        # Test tone checkbox
        self.test_tones = tk.BooleanVar(value=False)
        test_check = ttk.Checkbutton(buffer_frame, text="Include test tones", variable=self.test_tones)
        test_check.pack(side=tk.LEFT, padx=20)
    
    def _create_log_area(self):
        """Create log text area"""
        log_frame = ttk.LabelFrame(self.main_frame, text="Log", padding="5")
        log_frame.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.log_text = tk.Text(log_frame, height=10, wrap=tk.WORD)
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        log_scrollbar = ttk.Scrollbar(log_frame, orient=tk.VERTICAL, command=self.log_text.yview)
        log_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.log_text.config(yscrollcommand=log_scrollbar.set)
    
    def load_participant_files(self):
        """Load participant files from the directory"""
        self.folder_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
        self.participant_files = []
        
        debug_print(f"Loading participant files from {self.folder_path}", 0, self.log_text)
        
        try:
            # Get all WAV files
            all_files = [f for f in os.listdir(self.folder_path) if f.endswith('.wav')]
            
            # Sort by participant number
            def extract_participant_num(filename):
                match = re.search(r'participant_(\d+)_combined\.wav', filename)
                if match:
                    try:
                        return int(match.group(1))
                    except:
                        return 999  # High number for sorting if can't convert
                return 999
            
            all_files.sort(key=extract_participant_num)
            self.participant_files = all_files
            
            # Add to dropdown
            self.participant_dropdown['values'] = self.participant_files
            if self.participant_files:
                self.participant_dropdown.current(0)
            
            debug_print(f"Loaded {len(all_files)} participant files", 0, self.log_text)
            
        except Exception as e:
            debug_print(f"Error loading participant files: {e}", 0, self.log_text)
            messagebox.showerror("Error", f"Could not load participant files: {e}")
    
    def get_start_time(self):
        """Get the start time in seconds from the minute/second inputs"""
        try:
            minutes = int(self.minutes_var.get())
            seconds = int(self.seconds_var.get())
            return minutes * 60 + seconds
        except ValueError:
            return 0
    
    def start_playback(self):
        """Start playing the selected audio file"""
        # Check if a file is selected
        if not self.participant_var.get():
            messagebox.showinfo("Selection Required", "Please select a participant file to play.")
            return
        
        # Get file path
        file_path = os.path.join(self.folder_path, self.participant_var.get())
        
        # Get buffer size
        try:
            buffer_size = int(self.buffer_size.get())
        except:
            buffer_size = 512  # Default
        
        # Get start time
        start_time = self.get_start_time()
        
        # Clear log
        self.log_text.delete(1.0, tk.END)
        debug_print(f"Playing file: {self.participant_var.get()}", 0, self.log_text)
        debug_print(f"Buffer size: {buffer_size}", 0, self.log_text)
        debug_print(f"Start time: {start_time} seconds", 0, self.log_text)
        
        # Update UI
        self.play_button.config(state=tk.DISABLED)
        self.pause_button.config(state=tk.NORMAL)
        self.stop_button.config(state=tk.NORMAL)
        self.restart_button.config(state=tk.NORMAL)
        self.status_var.set(f"Loading: {self.participant_var.get()}")
        
        # Load and prepare in a separate thread
        def load_and_play():
            # Load the file
            if not self.audio_player.load_file(file_path, self.log_text):
                self.root.after(0, lambda: self.status_var.set("Error loading file"))
                self.root.after(0, lambda: self.play_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.pause_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.restart_button.config(state=tk.DISABLED))
                return
            
            # Prepare for playback
            if not self.audio_player.prepare_playback(
                self.log_text, buffer_size=buffer_size, test_tones=self.test_tones.get()):
                self.root.after(0, lambda: self.status_var.set("Error preparing playback"))
                self.root.after(0, lambda: self.play_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.pause_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.restart_button.config(state=tk.DISABLED))
                return
            
            # Start playback
            if not self.audio_player.start_playback(self.log_text, buffer_size=buffer_size, start_time=start_time):
                self.root.after(0, lambda: self.status_var.set("Error starting playback"))
                self.root.after(0, lambda: self.play_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.pause_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.restart_button.config(state=tk.DISABLED))
                return
            
            # Start progress updates
            self.root.after(0, self.update_progress)
            self.root.after(0, lambda: self.status_var.set(f"Playing: {self.participant_var.get()}"))
        
        threading.Thread(target=load_and_play, daemon=True).start()
    
    def pause_resume(self):
        """Pause or resume playback"""
        if not self.audio_player.is_playing:
            return
            
        if self.audio_player.is_paused:
            if self.audio_player.resume_playback(self.log_text):
                self.pause_button.config(text="Pause")
                self.status_var.set(f"Playing: {self.participant_var.get()}")
        else:
            if self.audio_player.pause_playback(self.log_text):
                self.pause_button.config(text="Resume")
                self.status_var.set(f"Paused: {self.participant_var.get()}")
    
    def stop_playback(self):
        """Stop playback"""
        if self.audio_player.stop_playback(self.log_text):
            self.reset_ui()
            self.status_var.set("Stopped")
    
    def restart_playback(self):
        """Restart playback from the beginning"""
        if self.audio_player.is_playing:
            self.audio_player.stop_playback(self.log_text)
            
        # Reset time input
        self.minutes_var.set("00")
        self.seconds_var.set("00")
        
        # Start playback
        self.start_playback()
    
    def reset_ui(self):
        """Reset UI elements after playback finishes"""
        self.play_button.config(state=tk.NORMAL)
        self.pause_button.config(state=tk.DISABLED, text="Pause")
        self.stop_button.config(state=tk.DISABLED)
        self.restart_button.config(state=tk.DISABLED)
        
        # Cancel progress updates
        if self.update_progress_id is not None:
            self.root.after_cancel(self.update_progress_id)
            self.update_progress_id = None
    
    def update_progress(self):
        """Update the progress display"""
        if not self.audio_player.is_playing:
            self.reset_ui()
            self.progress_var.set("00:00 / 00:00")
            return
        
        # Get current position
        position = self.audio_player.get_position()
        duration = self.audio_player.get_duration()
        
        # Format time
        pos_min = int(position) // 60
        pos_sec = int(position) % 60
        dur_min = int(duration) // 60
        dur_sec = int(duration) % 60
        
        self.progress_var.set(f"{pos_min:02d}:{pos_sec:02d} / {dur_min:02d}:{dur_sec:02d}")
        
        # Schedule next update
        self.update_progress_id = self.root.after(500, self.update_progress)
        
        # Check if playback finished
        if not self.audio_player.is_playing:
            self.reset_ui()
            self.status_var.set("Ready")

# Run the application
if __name__ == "__main__":
    root = tk.Tk()
    app = AudioPlayerApp(root)
    root.mainloop()

In [2]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import time
import threading
import tkinter as tk
from tkinter import ttk, messagebox
import os
import re

# Configure sounddevice for low latency
sd.default.latency = 'low'

class AudioPlayer:
    def __init__(self):
        self.audio_data = None
        self.sample_rate = 0
        self.playback_thread = None
        self.is_playing = False
        self.is_paused = False
        self.current_position = 0
        self.output_12_id = None
        self.output_34_id = None
        self.left_stream = None
        self.right_stream = None
        self.stop_event = threading.Event()
        self.pause_event = threading.Event()

    def load_file(self, file_path, text_widget=None):
        """Load audio file and prepare for playback"""
        try:
            debug_print(f"Loading file: {file_path}", 1, text_widget)
            audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
            debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1, text_widget)
            
            self.audio_data = audio
            self.sample_rate = file_sr
            self.current_position = 0
            return True
        except Exception as e:
            debug_print(f"Error loading audio file: {e}", 1, text_widget)
            return False

    def prepare_playback(self, text_widget=None, buffer_size=512, test_tones=False):
        """Prepare audio devices and data for playback"""
        # Get device list
        devices = sd.query_devices()
        
        # Look for Komplete Audio ASIO devices
        self.output_12_id = None
        self.output_34_id = None
        
        for i, device in enumerate(devices):
            if device['max_output_channels'] > 0:
                # Check if it's an ASIO device
                if 'hostapi' in device and device['hostapi'] == 3:  # ASIO
                    name = device['name'].lower()
                    debug_print(f"Found ASIO device {i}: {device['name']}", 1, text_widget)
                    if "output 1/2" in name and ("komplete" in name or "1/2" in name):
                        self.output_12_id = i
                        debug_print(f"Found Output 1/2 ASIO device: ID {i} - {device['name']}", 1, text_widget)
                    elif "output 3/4" in name and ("komplete" in name or "3/4" in name):
                        self.output_34_id = i
                        debug_print(f"Found Output 3/4 ASIO device: ID {i} - {device['name']}", 1, text_widget)
        
        # Check if we found the devices
        if self.output_12_id is None or self.output_34_id is None:
            debug_print("Could not find both ASIO devices for Komplete Audio", 1, text_widget)
            debug_print("Looking for any suitable multi-channel device instead", 1, text_widget)
            
            # Try to find any multi-channel device as a fallback
            for i, device in enumerate(devices):
                if device['max_output_channels'] >= 4 and 'hostapi' in device and device['hostapi'] == 3:
                    debug_print(f"Found multi-channel ASIO device: {device['name']}", 1, text_widget)
                    self.output_12_id = i  # Use the same device for both outputs
                    self.output_34_id = i
                    break
            
            if self.output_12_id is None:
                debug_print("No suitable audio device found.", 1, text_widget)
                return False
        
        # Get device sample rate
        try:
            device_info = sd.query_devices(self.output_12_id)
            device_sr = int(device_info['default_samplerate'])
        except:
            device_sr = 44100  # Default to 44.1kHz if query fails
        
        debug_print(f"Using sample rate: {device_sr}Hz", 1, text_widget)
        debug_print(f"Using buffer size: {buffer_size} samples", 1, text_widget)
        
        # Resample if needed
        if self.sample_rate != device_sr:
            debug_print(f"Sample rate mismatch: file={self.sample_rate}Hz, device={device_sr}Hz", 1, text_widget)
            self.audio_data = simple_resample(self.audio_data, self.sample_rate, device_sr, text_widget)
            self.sample_rate = device_sr
        
        # Add test signals to beginning if requested
        if test_tones:
            test_duration = 3  # seconds
            test_samples = int(test_duration * device_sr)
            
            t = np.linspace(0, test_duration, test_samples, endpoint=False)
            test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
            test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
            
            # Add fade in/out
            fade_samples = int(0.1 * device_sr)
            fade_in = np.linspace(0, 1, fade_samples)
            fade_out = np.linspace(1, 0, fade_samples)
            
            test_left[:fade_samples] *= fade_in
            test_left[-fade_samples:] *= fade_out
            test_right[:fade_samples] *= fade_in
            test_right[-fade_samples:] *= fade_out
            
            # Create a new longer audio array to PREPEND (not replace) test tones
            new_length = len(self.audio_data) + test_samples
            new_audio = np.zeros((new_length, 2), dtype=np.float32)
            
            # Place test tones at the beginning
            new_audio[:test_samples, 0] = test_left
            new_audio[:test_samples, 1] = test_right
            
            # Place original audio after the test tones
            new_audio[test_samples:, :] = self.audio_data
            
            # Replace audio data with the new combined version
            self.audio_data = new_audio
            debug_print(f"Prepended test tones: 440Hz on left, 880Hz on right ({test_duration} seconds)", 1, text_widget)
        
        return True

    def start_playback(self, text_widget=None, buffer_size=512, start_time=0.0):
        """Start audio playback from specified time position"""
        if not self.audio_data is not None or self.is_playing:
            return False
            
        # Calculate start position in samples
        start_sample = int(start_time * self.sample_rate)
        if start_sample >= len(self.audio_data):
            start_sample = 0
            
        self.current_position = start_sample
        
        # Reset events
        self.stop_event.clear()
        self.pause_event.clear()
        
        # Split channels
        left_channel = self.audio_data[:, 0].copy()
        right_channel = self.audio_data[:, 1].copy()
        
        # Create stereo output for each device (we need to send stereo)
        left_output = np.column_stack((left_channel, left_channel)).astype(np.float32)
        right_output = np.column_stack((right_channel, right_channel)).astype(np.float32)
        
        # Start the playback thread
        self.playback_thread = threading.Thread(
            target=self._playback_thread_func,
            args=(left_output, right_output, buffer_size, start_sample, text_widget),
            daemon=True
        )
        
        self.is_playing = True
        self.is_paused = False
        self.playback_thread.start()
        
        debug_print(f"Playback started from position {start_time:.2f}s", 1, text_widget)
        return True
        
    def _playback_thread_func(self, left_output, right_output, buffer_size, start_sample, text_widget):
        """Thread function for audio playback"""
        try:
            # Truncate outputs to start at the desired position
            left_output = left_output[start_sample:]
            right_output = right_output[start_sample:]
            
            # Set up callback functions for streaming
            def left_callback(outdata, frames, time, status):
                if self.stop_event.is_set():
                    raise sd.CallbackStop
                
                if self.pause_event.is_set():
                    outdata.fill(0)
                    return
                    
                position = self.current_position
                end = position + frames
                
                if end > len(left_output):
                    outdata[:len(left_output) - position] = left_output[position:]
                    outdata[len(left_output) - position:] = 0
                    raise sd.CallbackStop
                else:
                    outdata[:] = left_output[position:end]
                    self.current_position = end
            
            def right_callback(outdata, frames, time, status):
                if self.stop_event.is_set():
                    raise sd.CallbackStop
                
                if self.pause_event.is_set():
                    outdata.fill(0)
                    return
                    
                position = self.current_position
                end = position + frames
                
                if end > len(right_output):
                    outdata[:len(right_output) - position] = right_output[position:]
                    outdata[len(right_output) - position:] = 0
                    raise sd.CallbackStop
                else:
                    outdata[:] = right_output[position:end]
            
            # Start the streams
            self.left_stream = sd.OutputStream(
                device=self.output_12_id,
                channels=2,
                callback=left_callback,
                samplerate=self.sample_rate,
                blocksize=buffer_size,
                dtype='float32'
            )
            
            self.right_stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=right_callback,
                samplerate=self.sample_rate,
                blocksize=buffer_size,
                dtype='float32'
            )
            
            self.left_stream.start()
            self.right_stream.start()
            
            # Wait for completion
            while self.left_stream.active or self.right_stream.active:
                time.sleep(0.1)
                if self.stop_event.is_set():
                    break
            
        except Exception as e:
            debug_print(f"Error in playback thread: {e}", 1, text_widget)
        
        finally:
            # Clean up
            if hasattr(self, 'left_stream') and self.left_stream:
                self.left_stream.stop()
                self.left_stream.close()
                self.left_stream = None
                
            if hasattr(self, 'right_stream') and self.right_stream:
                self.right_stream.stop()
                self.right_stream.close()
                self.right_stream = None
                
            self.is_playing = False
            self.is_paused = False
            
            # Signal in main thread that playback is complete
            if text_widget and text_widget.winfo_exists():
                text_widget.after(0, lambda: debug_print("Playback complete", 1, text_widget))
    
    def pause_playback(self, text_widget=None):
        """Pause audio playback"""
        if self.is_playing and not self.is_paused:
            self.pause_event.set()
            self.is_paused = True
            debug_print("Playback paused", 1, text_widget)
            return True
        return False
    
    def resume_playback(self, text_widget=None):
        """Resume audio playback"""
        if self.is_playing and self.is_paused:
            self.pause_event.clear()
            self.is_paused = False
            debug_print("Playback resumed", 1, text_widget)
            return True
        return False
    
    def stop_playback(self, text_widget=None):
        """Stop audio playback"""
        if self.is_playing:
            self.stop_event.set()
            
            # Close streams
            if hasattr(self, 'left_stream') and self.left_stream:
                try:
                    self.left_stream.stop()
                    self.left_stream.close()
                except:
                    pass
                self.left_stream = None
                
            if hasattr(self, 'right_stream') and self.right_stream:
                try:
                    self.right_stream.stop()
                    self.right_stream.close()
                except:
                    pass
                self.right_stream = None
                
            # Wait for thread to finish
            if self.playback_thread and self.playback_thread.is_alive():
                self.playback_thread.join(timeout=1.0)
                
            self.is_playing = False
            self.is_paused = False
            debug_print("Playback stopped", 1, text_widget)
            return True
        return False

    def get_position(self):
        """Get current playback position in seconds"""
        if self.sample_rate > 0:
            return self.current_position / self.sample_rate
        return 0.0
    
    def get_duration(self):
        """Get audio duration in seconds"""
        if self.audio_data is not None and self.sample_rate > 0:
            return len(self.audio_data) / self.sample_rate
        return 0.0

def debug_print(message, indent=0, text_widget=None):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    log_message = f"[{timestamp}] {indent_str}{message}"
    print(log_message)
    
    # If we have a text widget, also log there
    if text_widget:
        text_widget.insert(tk.END, log_message + "\n")
        text_widget.see(tk.END)  # Auto-scroll to end

def simple_resample(audio, orig_sr, target_sr, text_widget=None):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1, text_widget)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1, text_widget)
    return resampled

class AudioPlayerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Breathing Space Audio Player")
        self.root.geometry("600x500")  # More compact size
        
        # Set up window close handler
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        # Create audio player instance
        self.audio_player = AudioPlayer()
        
        # Create main frame with padding
        self.main_frame = ttk.Frame(root, padding="5")
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Create a style
        self.style = ttk.Style()
        self.style.configure("TButton", padding=2)
        self.style.configure("TFrame", padding=2)
        
        # Create UI
        self._create_participant_frame()
        self._create_playback_controls()
        self._create_options_frame()
        self._create_log_area()
        
        # Status bar
        self.status_var = tk.StringVar(value="Ready")
        self.status_bar = ttk.Label(root, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)
        
        # Load participant files
        self.load_participant_files()
        
        # Setup progress updating
        self.update_progress_id = None
    
    def _create_participant_frame(self):
        """Create the participant selection area"""
        select_frame = ttk.LabelFrame(self.main_frame, text="Participant Selection", padding="5")
        select_frame.pack(fill=tk.X, pady=5)
        
        # Participant dropdown
        ttk.Label(select_frame, text="Select Participant:").grid(row=0, column=0, padx=5, pady=2, sticky=tk.W)
        
        self.participant_var = tk.StringVar()
        self.participant_dropdown = ttk.Combobox(select_frame, textvariable=self.participant_var, state="readonly", width=40)
        self.participant_dropdown.grid(row=0, column=1, padx=5, pady=2, sticky=tk.W+tk.E)
    
    def _create_playback_controls(self):
        """Create playback controls section"""
        controls_frame = ttk.LabelFrame(self.main_frame, text="Playback Controls", padding="5")
        controls_frame.pack(fill=tk.X, pady=5)
        
        # Time input
        time_frame = ttk.Frame(controls_frame)
        time_frame.grid(row=0, column=0, columnspan=2, padx=5, pady=2, sticky=tk.W)
        
        ttk.Label(time_frame, text="Start Time (mm:ss):").pack(side=tk.LEFT, padx=5)
        
        self.minutes_var = tk.StringVar(value="00")
        self.seconds_var = tk.StringVar(value="00")
        
        minutes_entry = ttk.Entry(time_frame, textvariable=self.minutes_var, width=3)
        minutes_entry.pack(side=tk.LEFT)
        
        ttk.Label(time_frame, text=":").pack(side=tk.LEFT)
        
        seconds_entry = ttk.Entry(time_frame, textvariable=self.seconds_var, width=3)
        seconds_entry.pack(side=tk.LEFT, padx=(0, 10))
        
        # Progress display
        self.progress_var = tk.StringVar(value="00:00 / 00:00")
        progress_label = ttk.Label(time_frame, textvariable=self.progress_var)
        progress_label.pack(side=tk.LEFT, padx=10)
        
        # Control buttons
        button_frame = ttk.Frame(controls_frame)
        button_frame.grid(row=1, column=0, columnspan=2, padx=5, pady=5, sticky=tk.W+tk.E)
        
        self.play_button = ttk.Button(button_frame, text="Play", command=self.start_playback, width=8)
        self.play_button.pack(side=tk.LEFT, padx=2)
        
        self.pause_button = ttk.Button(button_frame, text="Pause", command=self.pause_resume, width=8)
        self.pause_button.pack(side=tk.LEFT, padx=2)
        self.pause_button.config(state=tk.DISABLED)
        
        self.stop_button = ttk.Button(button_frame, text="Stop", command=self.stop_playback, width=8)
        self.stop_button.pack(side=tk.LEFT, padx=2)
        self.stop_button.config(state=tk.DISABLED)
        
        self.restart_button = ttk.Button(button_frame, text="Restart", command=self.restart_playback, width=8)
        self.restart_button.pack(side=tk.LEFT, padx=2)
        self.restart_button.config(state=tk.DISABLED)
    
    def _create_options_frame(self):
        """Create playback options section"""
        options_frame = ttk.LabelFrame(self.main_frame, text="Playback Options", padding="5")
        options_frame.pack(fill=tk.X, pady=5)
        
        # Buffer size
        buffer_frame = ttk.Frame(options_frame)
        buffer_frame.pack(fill=tk.X)
        
        ttk.Label(buffer_frame, text="Buffer Size:").pack(side=tk.LEFT, padx=5)
        
        self.buffer_size = tk.StringVar(value="128")
        buffer_sizes = ["128", "256", "512", "1024", "2048"]
        buffer_combo = ttk.Combobox(buffer_frame, textvariable=self.buffer_size, values=buffer_sizes, width=8, state="readonly")
        buffer_combo.pack(side=tk.LEFT, padx=5)
        
        # Test tone checkbox
        self.test_tones = tk.BooleanVar(value=False)
        test_check = ttk.Checkbutton(buffer_frame, text="Include test tones", variable=self.test_tones)
        test_check.pack(side=tk.LEFT, padx=20)
    
    def _create_log_area(self):
        """Create log text area"""
        log_frame = ttk.LabelFrame(self.main_frame, text="Log", padding="5")
        log_frame.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.log_text = tk.Text(log_frame, height=10, wrap=tk.WORD)
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        log_scrollbar = ttk.Scrollbar(log_frame, orient=tk.VERTICAL, command=self.log_text.yview)
        log_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.log_text.config(yscrollcommand=log_scrollbar.set)
    
    def on_closing(self):
        """Handle window closing event"""
        debug_print("Application closing, stopping all audio playback...", 0, self.log_text)
        
        # Stop any ongoing playback
        if self.audio_player.is_playing:
            self.audio_player.stop_playback(self.log_text)
        
        # Add a small delay to ensure streams are closed
        time.sleep(0.2)
        
        # Force stop any remaining audio output
        try:
            sd.stop()
        except:
            pass
            
        # Close sounddevice streams if they exist
        if hasattr(self.audio_player, 'left_stream') and self.audio_player.left_stream:
            try:
                self.audio_player.left_stream.close()
            except:
                pass
                
        if hasattr(self.audio_player, 'right_stream') and self.audio_player.right_stream:
            try:
                self.audio_player.right_stream.close()
            except:
                pass
        
        # Destroy the window and exit
        self.root.destroy()
    
    def load_participant_files(self):
        """Load participant files from the directory"""
        self.folder_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
        self.participant_files = []
        
        debug_print(f"Loading participant files from {self.folder_path}", 0, self.log_text)
        
        try:
            # Get all WAV files
            all_files = [f for f in os.listdir(self.folder_path) if f.endswith('.wav')]
            
            # Sort by participant number
            def extract_participant_num(filename):
                match = re.search(r'participant_(\d+)_combined\.wav', filename)
                if match:
                    try:
                        return int(match.group(1))
                    except:
                        return 999  # High number for sorting if can't convert
                return 999
            
            all_files.sort(key=extract_participant_num)
            self.participant_files = all_files
            
            # Add to dropdown
            self.participant_dropdown['values'] = self.participant_files
            if self.participant_files:
                self.participant_dropdown.current(0)
            
            debug_print(f"Loaded {len(all_files)} participant files", 0, self.log_text)
            
        except Exception as e:
            debug_print(f"Error loading participant files: {e}", 0, self.log_text)
            messagebox.showerror("Error", f"Could not load participant files: {e}")
    
    def get_start_time(self):
        """Get the start time in seconds from the minute/second inputs"""
        try:
            minutes = int(self.minutes_var.get())
            seconds = int(self.seconds_var.get())
            return minutes * 60 + seconds
        except ValueError:
            return 0
    
    def start_playback(self):
        """Start playing the selected audio file"""
        # Check if a file is selected
        if not self.participant_var.get():
            messagebox.showinfo("Selection Required", "Please select a participant file to play.")
            return
        
        # Get file path
        file_path = os.path.join(self.folder_path, self.participant_var.get())
        
        # Get buffer size
        try:
            buffer_size = int(self.buffer_size.get())
        except:
            buffer_size = 512  # Default
        
        # Get start time
        start_time = self.get_start_time()
        
        # Clear log
        self.log_text.delete(1.0, tk.END)
        debug_print(f"Playing file: {self.participant_var.get()}", 0, self.log_text)
        debug_print(f"Buffer size: {buffer_size}", 0, self.log_text)
        debug_print(f"Start time: {start_time} seconds", 0, self.log_text)
        
        # Update UI - All buttons should remain enabled except Play
        self.play_button.config(state=tk.DISABLED)
        self.pause_button.config(state=tk.NORMAL)
        self.stop_button.config(state=tk.NORMAL)
        self.restart_button.config(state=tk.NORMAL)
        self.status_var.set(f"Loading: {self.participant_var.get()}")
        
        # Load and prepare in a separate thread
        def load_and_play():
            # Load the file
            if not self.audio_player.load_file(file_path, self.log_text):
                self.root.after(0, lambda: self.status_var.set("Error loading file"))
                self.root.after(0, lambda: self.play_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.pause_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.restart_button.config(state=tk.DISABLED))
                return
            
            # Prepare for playback
            if not self.audio_player.prepare_playback(
                self.log_text, buffer_size=buffer_size, test_tones=self.test_tones.get()):
                self.root.after(0, lambda: self.status_var.set("Error preparing playback"))
                self.root.after(0, lambda: self.play_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.pause_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.restart_button.config(state=tk.DISABLED))
                return
            
            # Start playback
            if not self.audio_player.start_playback(self.log_text, buffer_size=buffer_size, start_time=start_time):
                self.root.after(0, lambda: self.status_var.set("Error starting playback"))
                self.root.after(0, lambda: self.play_button.config(state=tk.NORMAL))
                self.root.after(0, lambda: self.pause_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.stop_button.config(state=tk.DISABLED))
                self.root.after(0, lambda: self.restart_button.config(state=tk.DISABLED))
                return
            
            # Start progress updates
            self.root.after(0, self.update_progress)
            self.root.after(0, lambda: self.status_var.set(f"Playing: {self.participant_var.get()}"))
        
        threading.Thread(target=load_and_play, daemon=True).start()
    
    def pause_resume(self):
        """Pause or resume playback"""
        if not self.audio_player.is_playing:
            return
            
        if self.audio_player.is_paused:
            if self.audio_player.resume_playback(self.log_text):
                self.pause_button.config(text="Pause")
                self.status_var.set(f"Playing: {self.participant_var.get()}")
        else:
            if self.audio_player.pause_playback(self.log_text):
                self.pause_button.config(text="Resume")
                self.status_var.set(f"Paused: {self.participant_var.get()}")
    
    def stop_playback(self):
        """Stop playback"""
        if self.audio_player.stop_playback(self.log_text):
            self.reset_ui()
            self.status_var.set("Stopped")
    
    def restart_playback(self):
        """Restart playback from the beginning"""
        if self.audio_player.is_playing:
            self.audio_player.stop_playback(self.log_text)
            
        # Reset time input
        self.minutes_var.set("00")
        self.seconds_var.set("00")
        
        # Start playback
        self.start_playback()
    
    def reset_ui(self):
        """Reset UI elements after playback finishes"""
        self.play_button.config(state=tk.NORMAL)
        self.pause_button.config(state=tk.DISABLED, text="Pause")
        self.stop_button.config(state=tk.DISABLED)
        self.restart_button.config(state=tk.DISABLED)
        
        # Cancel progress updates
        if self.update_progress_id is not None:
            self.root.after_cancel(self.update_progress_id)
            self.update_progress_id = None
    
    def update_progress(self):
        """Update the progress display"""
        if not self.audio_player.is_playing:
            self.reset_ui()
            self.progress_var.set("00:00 / 00:00")
            return
        
        # Get current position
        position = self.audio_player.get_position()
        duration = self.audio_player.get_duration()
        
        # Format time
        pos_min = int(position) // 60
        pos_sec = int(position) % 60
        dur_min = int(duration) // 60
        dur_sec = int(duration) % 60
        
        self.progress_var.set(f"{pos_min:02d}:{pos_sec:02d} / {dur_min:02d}:{dur_sec:02d}")
        
        # Schedule next update
        self.update_progress_id = self.root.after(500, self.update_progress)
        
        # Check if playback finished
        if not self.audio_player.is_playing:
            self.reset_ui()
            self.status_var.set("Ready")

# Run the application
if __name__ == "__main__":
    # Clean up on startup just in case
    try:
        sd.stop()
    except:
        pass
        
    root = tk.Tk()
    app = AudioPlayerApp(root)
    root.mainloop()
    
    # Ensure all audio is stopped after mainloop ends
    try:
        sd.stop()
    except:
        pass

[21:16:30] Loading participant files from C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch
[21:16:30] Loaded 100 participant files
[21:16:36] Playing file: participant_00_combined.wav
[21:16:36] Buffer size: 128
[21:16:36] Start time: 0 seconds
[21:16:36]   Loading file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch\participant_00_combined.wav
[21:16:37]   File loaded: 73879143 samples, 44100Hz, 2 channels
[21:16:37]   Found ASIO device 28: Speakers (Nahimic mirroring Wave Speaker)
[21:16:37]   Found ASIO device 29: Speakers (Realtek HD Audio output with SST)
[21:16:37]   Found ASIO device 33: Headphones (Realtek HD Audio 2nd output with SST)
[21:16:37]   Found ASIO device 34: Speakers ()
[21:16:37]   Found ASIO device 35: Speakers (Nahimic Easy Surround)
[21:16:37]   Found ASIO device 39: Speakers ()
[21:16:37]   Found ASIO device 40: SPDIF Output (SPDIF Output)
[21:16:37]   Found ASIO d

In [1]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import time
import threading
import tkinter as tk
from tkinter import ttk, messagebox
import os
import re

# Configure sounddevice for low latency
sd.default.latency = 'low'

class AudioPlayer:
    def __init__(self):
        self.reset()
        
    def reset(self):
        """Reset all player state"""
        # Stop any active playback first
        self.stop_playback()
        
        # Reset all instance variables
        self.audio_data = None
        self.sample_rate = 0
        self.playback_thread = None
        self.is_playing = False
        self.is_paused = False
        self.current_position = 0
        self.output_12_id = None
        self.output_34_id = None
        self.left_stream = None
        self.right_stream = None
        self.stop_event = threading.Event()
        self.pause_event = threading.Event()
        
        # Force stop any lingering audio
        try:
            sd.stop()
        except:
            pass

    def load_file(self, file_path, text_widget=None):
        """Load audio file and prepare for playback"""
        # First, reset the player state
        self.reset()
        
        try:
            debug_print(f"Loading file: {file_path}", 1, text_widget)
            audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
            debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1, text_widget)
            
            self.audio_data = audio
            self.sample_rate = file_sr
            self.current_position = 0
            return True
        except Exception as e:
            debug_print(f"Error loading audio file: {e}", 1, text_widget)
            self.reset()  # Reset on error
            return False

    def prepare_playback(self, text_widget=None, buffer_size=512, test_tones=False):
        """Prepare audio devices and data for playback"""
        # Get device list
        devices = sd.query_devices()
        
        # Look for Komplete Audio ASIO devices
        self.output_12_id = None
        self.output_34_id = None
        
        for i, device in enumerate(devices):
            if device['max_output_channels'] > 0:
                # Check if it's an ASIO device
                if 'hostapi' in device and device['hostapi'] == 3:  # ASIO
                    name = device['name'].lower()
                    debug_print(f"Found ASIO device {i}: {device['name']}", 1, text_widget)
                    if "output 1/2" in name and ("komplete" in name or "1/2" in name):
                        self.output_12_id = i
                        debug_print(f"Found Output 1/2 ASIO device: ID {i} - {device['name']}", 1, text_widget)
                    elif "output 3/4" in name and ("komplete" in name or "3/4" in name):
                        self.output_34_id = i
                        debug_print(f"Found Output 3/4 ASIO device: ID {i} - {device['name']}", 1, text_widget)
        
        # Check if we found the devices
        if self.output_12_id is None or self.output_34_id is None:
            debug_print("Could not find both ASIO devices for Komplete Audio", 1, text_widget)
            debug_print("Looking for any suitable multi-channel device instead", 1, text_widget)
            
            # Try to find any multi-channel device as a fallback
            for i, device in enumerate(devices):
                if device['max_output_channels'] >= 4 and 'hostapi' in device and device['hostapi'] == 3:
                    debug_print(f"Found multi-channel ASIO device: {device['name']}", 1, text_widget)
                    self.output_12_id = i  # Use the same device for both outputs
                    self.output_34_id = i
                    break
            
            if self.output_12_id is None:
                debug_print("No suitable audio device found.", 1, text_widget)
                return False
        
        # Get device sample rate
        try:
            device_info = sd.query_devices(self.output_12_id)
            device_sr = int(device_info['default_samplerate'])
        except:
            device_sr = 44100  # Default to 44.1kHz if query fails
        
        debug_print(f"Using sample rate: {device_sr}Hz", 1, text_widget)
        debug_print(f"Using buffer size: {buffer_size} samples", 1, text_widget)
        
        # Resample if needed
        if self.sample_rate != device_sr:
            debug_print(f"Sample rate mismatch: file={self.sample_rate}Hz, device={device_sr}Hz", 1, text_widget)
            self.audio_data = simple_resample(self.audio_data, self.sample_rate, device_sr, text_widget)
            self.sample_rate = device_sr
        
        # Add test signals to beginning if requested
        if test_tones:
            test_duration = 3  # seconds
            test_samples = int(test_duration * device_sr)
            
            t = np.linspace(0, test_duration, test_samples, endpoint=False)
            test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
            test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
            
            # Add fade in/out
            fade_samples = int(0.1 * device_sr)
            fade_in = np.linspace(0, 1, fade_samples)
            fade_out = np.linspace(1, 0, fade_samples)
            
            test_left[:fade_samples] *= fade_in
            test_left[-fade_samples:] *= fade_out
            test_right[:fade_samples] *= fade_in
            test_right[-fade_samples:] *= fade_out
            
            # Create a new longer audio array to PREPEND (not replace) test tones
            new_length = len(self.audio_data) + test_samples
            new_audio = np.zeros((new_length, 2), dtype=np.float32)
            
            # Place test tones at the beginning
            new_audio[:test_samples, 0] = test_left
            new_audio[:test_samples, 1] = test_right
            
            # Place original audio after the test tones
            new_audio[test_samples:, :] = self.audio_data
            
            # Replace audio data with the new combined version
            self.audio_data = new_audio
            debug_print(f"Prepended test tones: 440Hz on left, 880Hz on right ({test_duration} seconds)", 1, text_widget)
        
        return True

    def start_playback(self, text_widget=None, buffer_size=512, start_time=0.0):
        """Start audio playback from specified time position"""
        # First stop any existing playback
        if self.is_playing:
            self.stop_playback(text_widget)
            # Short delay to ensure cleanup is complete
            time.sleep(0.1)
            
        if self.audio_data is None:
            debug_print("No audio data loaded", 1, text_widget)
            return False
            
        # Calculate start position in samples
        start_sample = int(start_time * self.sample_rate)
        if start_sample >= len(self.audio_data):
            debug_print(f"Start time ({start_time}s) exceeds audio duration", 1, text_widget)
            start_sample = 0
            
        self.current_position = start_sample
        
        # Reset events
        self.stop_event.clear()
        self.pause_event.clear()
        
        # Split channels
        try:
            left_channel = self.audio_data[:, 0].copy()
            right_channel = self.audio_data[:, 1].copy()
            
            # Create stereo output for each device (we need to send stereo)
            left_output = np.column_stack((left_channel, left_channel)).astype(np.float32)
            right_output = np.column_stack((right_channel, right_channel)).astype(np.float32)
            
            # Start the playback thread
            self.playback_thread = threading.Thread(
                target=self._playback_thread_func,
                args=(left_output, right_output, buffer_size, start_sample, text_widget),
                daemon=True
            )
            
            self.is_playing = True
            self.is_paused = False
            self.playback_thread.start()
            
            debug_print(f"Playback started from position {start_time:.2f}s", 1, text_widget)
            return True
        except Exception as e:
            debug_print(f"Error starting playback: {e}", 1, text_widget)
            self.is_playing = False
            self.is_paused = False
            return False
        
    def _playback_thread_func(self, left_output, right_output, buffer_size, start_sample, text_widget):
        """Thread function for audio playback"""
        try:
            # Truncate outputs to start at the desired position
            left_output = left_output[start_sample:]
            right_output = right_output[start_sample:]
            
            # Set up callback functions for streaming
            def left_callback(outdata, frames, time, status):
                if self.stop_event.is_set():
                    raise sd.CallbackStop
                
                if self.pause_event.is_set():
                    outdata.fill(0)
                    return
                    
                position = self.current_position
                end = position + frames
                
                if end > len(left_output):
                    outdata[:len(left_output) - position] = left_output[position:]
                    outdata[len(left_output) - position:] = 0
                    raise sd.CallbackStop
                else:
                    outdata[:] = left_output[position:end]
                    self.current_position = end
            
            def right_callback(outdata, frames, time, status):
                if self.stop_event.is_set():
                    raise sd.CallbackStop
                
                if self.pause_event.is_set():
                    outdata.fill(0)
                    return
                    
                position = self.current_position
                end = position + frames
                
                if end > len(right_output):
                    outdata[:len(right_output) - position] = right_output[position:]
                    outdata[len(right_output) - position:] = 0
                    raise sd.CallbackStop
                else:
                    outdata[:] = right_output[position:end]
            
            # Start the streams
            self.left_stream = sd.OutputStream(
                device=self.output_12_id,
                channels=2,
                callback=left_callback,
                samplerate=self.sample_rate,
                blocksize=buffer_size,
                dtype='float32'
            )
            
            self.right_stream = sd.OutputStream(
                device=self.output_34_id,
                channels=2,
                callback=right_callback,
                samplerate=self.sample_rate,
                blocksize=buffer_size,
                dtype='float32'
            )
            
            self.left_stream.start()
            self.right_stream.start()
            
            # Wait for completion
            while self.left_stream.active or self.right_stream.active:
                time.sleep(0.1)
                if self.stop_event.is_set():
                    break
            
        except Exception as e:
            debug_print(f"Error in playback thread: {e}", 1, text_widget)
        
        finally:
            # Clean up
            if hasattr(self, 'left_stream') and self.left_stream:
                self.left_stream.stop()
                self.left_stream.close()
                self.left_stream = None
                
            if hasattr(self, 'right_stream') and self.right_stream:
                self.right_stream.stop()
                self.right_stream.close()
                self.right_stream = None
                
            self.is_playing = False
            self.is_paused = False
            
            # Signal in main thread that playback is complete
            if text_widget and text_widget.winfo_exists():
                text_widget.after(0, lambda: debug_print("Playback complete", 1, text_widget))
    
    def pause_playback(self, text_widget=None):
        """Pause audio playback"""
        if self.is_playing and not self.is_paused:
            self.pause_event.set()
            self.is_paused = True
            debug_print("Playback paused", 1, text_widget)
            return True
        return False
    
    def resume_playback(self, text_widget=None):
        """Resume audio playback"""
        if self.is_playing and self.is_paused:
            self.pause_event.clear()
            self.is_paused = False
            debug_print("Playback resumed", 1, text_widget)
            return True
        return False
    
    def stop_playback(self, text_widget=None):
        """Stop audio playback"""
        if not hasattr(self, 'is_playing'):
            return False
            
        # Set stop flag even if not playing to ensure clean state
        if hasattr(self, 'stop_event'):
            self.stop_event.set()
            
        # Close streams
        if hasattr(self, 'left_stream') and self.left_stream:
            try:
                self.left_stream.stop()
                self.left_stream.close()
            except Exception as e:
                if text_widget:
                    debug_print(f"Error closing left stream: {e}", 1, text_widget)
            self.left_stream = None
                
        if hasattr(self, 'right_stream') and self.right_stream:
            try:
                self.right_stream.stop()
                self.right_stream.close()
            except Exception as e:
                if text_widget:
                    debug_print(f"Error closing right stream: {e}", 1, text_widget)
            self.right_stream = None
            
        # Force stop all sounddevice streams
        try:
            sd.stop()
        except:
            pass
            
        # Wait for thread to finish
        if hasattr(self, 'playback_thread') and self.playback_thread and self.playback_thread.is_alive():
            self.playback_thread.join(timeout=1.0)
        
        # Reset state
        was_playing = self.is_playing
        self.is_playing = False
        self.is_paused = False
        
        if was_playing and text_widget:
            debug_print("Playback stopped", 1, text_widget)
            
        return was_playing

    def get_position(self):
        """Get current playback position in seconds"""
        if self.sample_rate > 0:
            return self.current_position / self.sample_rate
        return 0.0
    
    def get_duration(self):
        """Get audio duration in seconds"""
        if self.audio_data is not None and self.sample_rate > 0:
            return len(self.audio_data) / self.sample_rate
        return 0.0

def debug_print(message, indent=0, text_widget=None):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    log_message = f"[{timestamp}] {indent_str}{message}"
    print(log_message)
    
    # If we have a text widget, also log there
    if text_widget:
        text_widget.insert(tk.END, log_message + "\n")
        text_widget.see(tk.END)  # Auto-scroll to end

def simple_resample(audio, orig_sr, target_sr, text_widget=None):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1, text_widget)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1, text_widget)
    return resampled

class AudioPlayerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Breathing Space Audio Player")
        self.root.geometry("600x500")  # More compact size
        
        # Set up window close handler
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)
        
        # Create audio player instance
        self.audio_player = AudioPlayer()
        
        # Create main frame with padding
        self.main_frame = ttk.Frame(root, padding="5")
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Create a style
        self.style = ttk.Style()
        self.style.configure("TButton", padding=2)
        self.style.configure("TFrame", padding=2)
        
        # Create UI
        self._create_participant_frame()
        self._create_playback_controls()
        self._create_options_frame()
        self._create_log_area()
        
        # Status bar
        self.status_var = tk.StringVar(value="Ready")
        self.status_bar = ttk.Label(root, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W)
        self.status_bar.pack(side=tk.BOTTOM, fill=tk.X)
        
        # Setup progress updating
        self.update_progress_id = None
        
        # Load participant files
        try:
            self.load_participant_files()
            debug_print("Application started successfully", 0, self.log_text)
        except Exception as e:
            debug_print(f"Error during initialization: {e}", 0, self.log_text)
            messagebox.showerror("Initialization Error", f"An error occurred during startup: {e}")
    
    def _create_participant_frame(self):
        """Create the participant selection area"""
        select_frame = ttk.LabelFrame(self.main_frame, text="Participant Selection", padding="5")
        select_frame.pack(fill=tk.X, pady=5)
        
        # Participant dropdown
        ttk.Label(select_frame, text="Select Participant:").grid(row=0, column=0, padx=5, pady=2, sticky=tk.W)
        
        self.participant_var = tk.StringVar()
        self.participant_dropdown = ttk.Combobox(select_frame, textvariable=self.participant_var, state="readonly", width=40)
        self.participant_dropdown.grid(row=0, column=1, padx=5, pady=2, sticky=tk.W+tk.E)
    
    def _create_playback_controls(self):
        """Create playback controls section"""
        controls_frame = ttk.LabelFrame(self.main_frame, text="Playback Controls", padding="5")
        controls_frame.pack(fill=tk.X, pady=5)
        
        # Time input
        time_frame = ttk.Frame(controls_frame)
        time_frame.grid(row=0, column=0, columnspan=2, padx=5, pady=2, sticky=tk.W)
        
        ttk.Label(time_frame, text="Start Time (mm:ss):").pack(side=tk.LEFT, padx=5)
        
        self.minutes_var = tk.StringVar(value="00")
        self.seconds_var = tk.StringVar(value="00")
        
        minutes_entry = ttk.Entry(time_frame, textvariable=self.minutes_var, width=3)
        minutes_entry.pack(side=tk.LEFT)
        
        ttk.Label(time_frame, text=":").pack(side=tk.LEFT)
        
        seconds_entry = ttk.Entry(time_frame, textvariable=self.seconds_var, width=3)
        seconds_entry.pack(side=tk.LEFT, padx=(0, 10))
        
        # Progress display
        self.progress_var = tk.StringVar(value="00:00 / 00:00")
        progress_label = ttk.Label(time_frame, textvariable=self.progress_var)
        progress_label.pack(side=tk.LEFT, padx=10)
        
        # Control buttons
        button_frame = ttk.Frame(controls_frame)
        button_frame.grid(row=1, column=0, columnspan=2, padx=5, pady=5, sticky=tk.W+tk.E)
        
        self.play_button = ttk.Button(button_frame, text="Play", command=self.start_playback, width=8)
        self.play_button.pack(side=tk.LEFT, padx=2)
        
        self.pause_button = ttk.Button(button_frame, text="Pause", command=self.pause_resume, width=8)
        self.pause_button.pack(side=tk.LEFT, padx=2)
        self.pause_button.config(state=tk.DISABLED)
        
        self.stop_button = ttk.Button(button_frame, text="Stop", command=self.stop_playback, width=8)
        self.stop_button.pack(side=tk.LEFT, padx=2)
        self.stop_button.config(state=tk.DISABLED)
        
        self.restart_button = ttk.Button(button_frame, text="Restart", command=self.restart_playback, width=8)
        self.restart_button.pack(side=tk.LEFT, padx=2)
        self.restart_button.config(state=tk.DISABLED)
    
    def _create_options_frame(self):
        """Create playback options section"""
        options_frame = ttk.LabelFrame(self.main_frame, text="Playback Options", padding="5")
        options_frame.pack(fill=tk.X, pady=5)
        
        # Buffer size
        buffer_frame = ttk.Frame(options_frame)
        buffer_frame.pack(fill=tk.X)
        
        ttk.Label(buffer_frame, text="Buffer Size:").pack(side=tk.LEFT, padx=5)
        
        self.buffer_size = tk.StringVar(value="128")
        buffer_sizes = ["128", "256", "512", "1024", "2048"]
        buffer_combo = ttk.Combobox(buffer_frame, textvariable=self.buffer_size, values=buffer_sizes, width=8, state="readonly")
        buffer_combo.pack(side=tk.LEFT, padx=5)
        
        # Test tone checkbox
        self.test_tones = tk.BooleanVar(value=False)
        test_check = ttk.Checkbutton(buffer_frame, text="Include test tones", variable=self.test_tones)
        test_check.pack(side=tk.LEFT, padx=20)
    
    def _create_log_area(self):
        """Create log text area"""
        log_frame = ttk.LabelFrame(self.main_frame, text="Log", padding="5")
        log_frame.pack(fill=tk.BOTH, expand=True, pady=5)
        
        self.log_text = tk.Text(log_frame, height=10, wrap=tk.WORD)
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        log_scrollbar = ttk.Scrollbar(log_frame, orient=tk.VERTICAL, command=self.log_text.yview)
        log_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.log_text.config(yscrollcommand=log_scrollbar.set)
    
    def on_closing(self):
        """Handle window closing event"""
        debug_print("Application closing, stopping all audio playback...", 0, self.log_text)
        
        # Cancel any scheduled updates
        if self.update_progress_id is not None:
            try:
                self.root.after_cancel(self.update_progress_id)
            except:
                pass
            self.update_progress_id = None
        
        # Stop any ongoing playback
        try:
            self.audio_player.reset()
        except:
            pass
        
        # Add a small delay to ensure streams are closed
        time.sleep(0.2)
        
        # Force stop any remaining audio output
        try:
            sd.stop()
        except:
            pass
        
        # Destroy the window and exit
        debug_print("Application closed.", 0, self.log_text)
        self.root.destroy()
    
    def load_participant_files(self):
        """Load participant files from the directory"""
        self.folder_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
        self.participant_files = []
        
        debug_print(f"Loading participant files from {self.folder_path}", 0, self.log_text)
        
        try:
            # Get all WAV files
            all_files = [f for f in os.listdir(self.folder_path) if f.endswith('.wav')]
            
            # Sort by participant number
            def extract_participant_num(filename):
                match = re.search(r'participant_(\d+)_combined\.wav', filename)
                if match:
                    try:
                        return int(match.group(1))
                    except:
                        return 999  # High number for sorting if can't convert
                return 999
            
            all_files.sort(key=extract_participant_num)
            self.participant_files = all_files
            
            # Add to dropdown
            self.participant_dropdown['values'] = self.participant_files
            if self.participant_files:
                self.participant_dropdown.current(0)
            
            debug_print(f"Loaded {len(all_files)} participant files", 0, self.log_text)
            
        except Exception as e:
            debug_print(f"Error loading participant files: {e}", 0, self.log_text)
            messagebox.showerror("Error", f"Could not load participant files: {e}")
    
    def get_start_time(self):
        """Get the start time in seconds from the minute/second inputs"""
        try:
            minutes = int(self.minutes_var.get())
            seconds = int(self.seconds_var.get())
            return minutes * 60 + seconds
        except ValueError:
            return 0
    
    def start_playback(self):
        """Start playing the selected audio file"""
        # Check if a file is selected
        if not self.participant_var.get():
            messagebox.showinfo("Selection Required", "Please select a participant file to play.")
            return
        
        # First, perform full reset to ensure clean state
        self.audio_player.reset()
        
        # Get file path
        file_path = os.path.join(self.folder_path, self.participant_var.get())
        
        # Get buffer size
        try:
            buffer_size = int(self.buffer_size.get())
        except:
            buffer_size = 128  # Default to smallest for best latency
        
        # Get start time
        start_time = self.get_start_time()
        
        # Clear log
        self.log_text.delete(1.0, tk.END)
        debug_print(f"Playing file: {self.participant_var.get()}", 0, self.log_text)
        debug_print(f"Buffer size: {buffer_size}", 0, self.log_text)
        debug_print(f"Start time: {start_time} seconds", 0, self.log_text)
        debug_print(f"Test tones: {'Enabled' if self.test_tones.get() else 'Disabled'}", 0, self.log_text)
        
        # Update UI - All buttons should remain enabled except Play
        self.play_button.config(state=tk.DISABLED)
        self.pause_button.config(state=tk.NORMAL, text="Pause")
        self.stop_button.config(state=tk.NORMAL)
        self.restart_button.config(state=tk.NORMAL)
        self.status_var.set(f"Loading: {self.participant_var.get()}")
        
        # Cancel any existing progress updates
        if self.update_progress_id is not None:
            self.root.after_cancel(self.update_progress_id)
            self.update_progress_id = None
        
        # Load and prepare in a separate thread
        def load_and_play():
            try:
                # Load the file
                if not self.audio_player.load_file(file_path, self.log_text):
                    self.root.after(0, lambda: self.status_var.set("Error loading file"))
                    self.root.after(0, lambda: self.reset_ui())
                    return
                
                # Prepare for playback
                if not self.audio_player.prepare_playback(
                    self.log_text, buffer_size=buffer_size, test_tones=self.test_tones.get()):
                    self.root.after(0, lambda: self.status_var.set("Error preparing playback"))
                    self.root.after(0, lambda: self.reset_ui())
                    return
                
                # Start playback
                if not self.audio_player.start_playback(self.log_text, buffer_size=buffer_size, start_time=start_time):
                    self.root.after(0, lambda: self.status_var.set("Error starting playback"))
                    self.root.after(0, lambda: self.reset_ui())
                    return
                
                # Start progress updates
                self.root.after(0, self.update_progress)
                self.root.after(0, lambda: self.status_var.set(f"Playing: {self.participant_var.get()}"))
            except Exception as e:
                self.root.after(0, lambda: debug_print(f"Error in playback thread: {e}", 0, self.log_text))
                self.root.after(0, lambda: self.reset_ui())
                self.root.after(0, lambda: self.status_var.set("Error occurred during playback"))
        
        threading.Thread(target=load_and_play, daemon=True).start()
    
    def pause_resume(self):
        """Pause or resume playback"""
        if not self.audio_player.is_playing:
            debug_print("Nothing playing to pause/resume", 0, self.log_text)
            return
            
        if self.audio_player.is_paused:
            if self.audio_player.resume_playback(self.log_text):
                self.pause_button.config(text="Pause")
                self.status_var.set(f"Playing: {self.participant_var.get()}")
                debug_print("Playback resumed", 0, self.log_text)
            else:
                # If resume fails, reset everything
                debug_print("Failed to resume playback, resetting", 0, self.log_text)
                self.stop_playback()
        else:
            if self.audio_player.pause_playback(self.log_text):
                self.pause_button.config(text="Resume")
                self.status_var.set(f"Paused: {self.participant_var.get()}")
                debug_print("Playback paused", 0, self.log_text)
            else:
                # If pause fails, reset everything
                debug_print("Failed to pause playback, resetting", 0, self.log_text)
                self.stop_playback()
    
    def stop_playback(self):
        """Stop playback"""
        if self.audio_player.stop_playback(self.log_text):
            # Full reset to ensure clean state
            self.audio_player.reset()
            self.reset_ui()
            self.status_var.set("Stopped")
            debug_print("Playback fully stopped and reset", 0, self.log_text)
    
    def restart_playback(self):
        """Restart playback from the beginning"""
        # Stop any current playback
        if self.audio_player.is_playing:
            self.audio_player.stop_playback(self.log_text)
        
        # Reset time input
        self.minutes_var.set("00")
        self.seconds_var.set("00")
        
        # Small delay to ensure everything is stopped
        time.sleep(0.1)
        
        # Start playback with a fresh state
        self.start_playback()
    
    def reset_ui(self):
        """Reset UI elements after playback finishes"""
        self.play_button.config(state=tk.NORMAL)
        self.pause_button.config(state=tk.DISABLED, text="Pause")
        self.stop_button.config(state=tk.DISABLED)
        self.restart_button.config(state=tk.DISABLED)
        
        # Cancel progress updates
        if self.update_progress_id is not None:
            self.root.after_cancel(self.update_progress_id)
            self.update_progress_id = None
    
    def update_progress(self):
        """Update the progress display"""
        # First check if player is still playing
        if not self.audio_player.is_playing:
            self.reset_ui()
            self.progress_var.set("00:00 / 00:00")
            return
        
        try:
            # Get current position
            position = self.audio_player.get_position()
            duration = self.audio_player.get_duration()
            
            # Format time
            pos_min = int(position) // 60
            pos_sec = int(position) % 60
            dur_min = int(duration) // 60
            dur_sec = int(duration) % 60
            
            self.progress_var.set(f"{pos_min:02d}:{pos_sec:02d} / {dur_min:02d}:{dur_sec:02d}")
            
            # Schedule next update - but don't accumulate if there are delays
            if self.update_progress_id is not None:
                self.root.after_cancel(self.update_progress_id)
            
            # Check again if still playing before scheduling next update
            if self.audio_player.is_playing:
                self.update_progress_id = self.root.after(200, self.update_progress)
            else:
                self.reset_ui()
                self.status_var.set("Ready")
                
        except Exception as e:
            debug_print(f"Error updating progress: {e}", 0, self.log_text)
            # Schedule recovery update
            self.update_progress_id = self.root.after(500, self.update_progress)

# Run the application
if __name__ == "__main__":
    # Clean up on startup just in case
    try:
        sd.stop()
    except:
        pass
    
    # Ensure sounddevice is configured for low latency
    sd.default.latency = 'low'
        
    root = tk.Tk()
    
    try:
        app = AudioPlayerApp(root)
        root.mainloop()
    except Exception as e:
        print(f"Critical error: {e}")
        # Attempt clean shutdown
        try:
            sd.stop()
        except:
            pass
    finally:
        # Final cleanup
        print("Application shutting down, performing final cleanup...")
        try:
            sd.stop()
        except:
            pass

[21:27:17] Loading participant files from C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch
[21:27:17] Loaded 100 participant files
[21:27:17] Application started successfully
[21:27:22] Playing file: participant_00_combined.wav
[21:27:22] Buffer size: 128
[21:27:22] Start time: 0 seconds
[21:27:22] Test tones: Disabled
[21:27:22]   Loading file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch\participant_00_combined.wav
[21:27:22]   File loaded: 73879143 samples, 44100Hz, 2 channels
[21:27:22]   Found ASIO device 28: Speakers (Nahimic mirroring Wave Speaker)
[21:27:22]   Found ASIO device 29: Speakers (Realtek HD Audio output with SST)
[21:27:22]   Found ASIO device 33: Headphones (Realtek HD Audio 2nd output with SST)
[21:27:22]   Found ASIO device 34: Speakers ()
[21:27:22]   Found ASIO device 35: Speakers (Nahimic Easy Surround)
[21:27:22]   Found ASIO device 39: Speakers ()
[21:27:22]  